# 第17章：单位根、协整与预测

从固定种子的构造机制出发，区分相关漂移水平与已知平稳价差，并分别使用普通ADF和Engle–Granger检验的校准。随后生成120个月的二维VAR，在前100个月拟合并递归预测两个月。默认只显示一张已有构造图；French的45对累计指数筛查及其历史图均跳过。累计指数代理也不等于可交易证券价格。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch17_time_series_ii_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import hashlib

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import statsmodels
from IPython.display import Image, display


from quant_course.cointegration import (
    adf_test,
    engle_granger_test,
    fit_var_forecast,
    screen_cointegration_pairs,
    simulate_cointegrated_pair,
    simulate_noncointegrated_common_trend,
    total_return_index_proxy,
    validate_monthly_frame,
)

print(f"Python={sys.version.split()[0]}; pandas={pd.__version__}; statsmodels={statsmodels.__version__}")

In [ ]:
# 固定种子构造：水平高相关但差仍是 I(1)，对照真实的已知平稳价差。
noncointegrated = simulate_noncointegrated_common_trend(n=900, seed=20261017)
cointegrated = simulate_cointegrated_pair(n=900, seed=20261018)
noncointegrated_eg = engle_granger_test(noncointegrated["x"], noncointegrated["y"])
cointegrated_eg = engle_granger_test(cointegrated["y"], cointegrated["x"])

print(f"共同趋势：corr(levels)={noncointegrated.x.corr(noncointegrated.y):.4f}; EG p={noncointegrated_eg.pvalue:.6f}")
print(f"协整构造：corr(levels)={cointegrated.x.corr(cointegrated.y):.4f}; EG p={cointegrated_eg.pvalue:.3e}")
print(f"协整 5% 专用临界值={cointegrated_eg.critical_values['5%']:.4f}")
assert noncointegrated.x.corr(noncointegrated.y) > 0.9
assert noncointegrated_eg.pvalue > 0.05
assert cointegrated_eg.pvalue < 0.01

## 手算回归到统计软件

若 (y_t=0.7y_{t-1}+\varepsilon_t)，相减得到 \(\Delta y_t=-0.3y_{t-1}+\varepsilon_t\)，所以 ADF 的简单 AR(1) 系数 \(\gamma=\phi-1=-0.3\)。ADF 的 p 值按指定确定项的非标准分布计算。

下面用前述模拟中的已知平稳价差跑普通单序列 ADF，再对估计的两条 I(1) 序列关系跑 EG。数值校准不同，因此读结果时不能互换两种 p 值。

In [ ]:
gamma = 0.7 - 1.0
spread_adf = adf_test(cointegrated["spread"], regression="c")
print(f"手算 gamma={gamma:.1f}")
print(f"已知生成的平稳 spread：ADF stat={spread_adf.statistic:.4f}, p={spread_adf.pvalue:.3e}, lag={spread_adf.used_lag}")
print(f"估计关系的残差检验：EG stat={cointegrated_eg.statistic:.4f}, p={cointegrated_eg.pvalue:.3e}")

## 二维 VAR：训练窗口与月份

这里生成平稳 VAR(1) 过程，把前 100 个月作为训练数据，递归预测未来两个月。这个演示对应章节 8 节的手算矩阵例；不同之处是代码从训练样本估计系数，预测期从训练末月之后开始。

In [ ]:
rng = np.random.default_rng(1704)
values = np.zeros((120, 2), dtype=float)
A = np.array([[0.35, 0.12], [0.05, 0.30]])
c = np.array([0.25, -0.10])
for t in range(1, len(values)):
    values[t] = c + A @ values[t - 1] + rng.normal(0.0, 0.10, size=2)
index = pd.period_range("1990-01", periods=len(values), freq="M", name="month")
var_data = pd.DataFrame(values, index=index, columns=["industry_A", "industry_B"])
train = validate_monthly_frame(var_data.iloc[:100])
var_prediction = fit_var_forecast(train, lags=1, steps=2)
print("training end:", train.index[-1])
print(var_prediction)
assert var_prediction.index.tolist() == [train.index[-1] + 1, train.index[-1] + 2]
assert var_prediction.columns.tolist() == ["industry_A", "industry_B"]

## 默认禁用的French累计指数筛查

历史加载器若获许可启用，会先核处理文件SHA-256，再把已为小数的收益复利为指数代理，不重复除以100。45对筛查仅使用截至2000年12月的观察。名义p值与BH调整q值不同，均不在这里选择可交易赢家。默认不运行该分支。

In [ ]:
if RUN_FRENCH:
    french_path = ROOT / "data/processed/french_10_industries_monthly.csv"
    expected_sha256 = "3b097d810c295d64eedfa1ece7067ff2a0707f5ca90e33935df9949251f70950"
    actual_sha256 = hashlib.sha256(french_path.read_bytes()).hexdigest()
    assert actual_sha256 == expected_sha256
    raw = pd.read_csv(french_path, dtype={"month": "string"})
    periods = pd.PeriodIndex(raw.pop("month"), freq="M", name="month")
    returns = validate_monthly_frame(pd.DataFrame(raw.to_numpy(dtype=float), index=periods, columns=raw.columns), min_observations=24)
    tr_index = total_return_index_proxy(returns)
    cutoff = pd.Period("2000-12", freq="M")
    pair_tests = screen_cointegration_pairs(tr_index, training_end=cutoff, trend="c")
    
    print(f"snapshot SHA-256={actual_sha256}")
    print(f"sample={returns.index[0]}–{returns.index[-1]}; n={len(returns)} months; pairs={len(pair_tests)}")
    print(f"training end={cutoff}; min p={pair_tests.pvalue.min():.6g}; median p={pair_tests.pvalue.median():.6g}")
    print(f"nominal p < .05={(pair_tests.pvalue < .05).sum()}; BH q < .05={(pair_tests.qvalue_bh < .05).sum()}")
    print("Exploratory current-vintage descriptive screen only; no selected pair is claimed tradable or profitable.")
    assert len(pair_tests) == 45
    assert pair_tests["training_end"].eq(str(cutoff)).all()
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 已有构造图

默认只显示图17-1：上排比较两条构造水平序列，下排观察价差是否仍漂移。图保存在figures/a4/，此处不重新生成。另一个French对数累计指数历史图及其首月100归一化说明属于被跳过的分支，不在默认展示中。

In [ ]:
display(Image(filename=str(ROOT / "fixtures/figures/a4/ch17_cointegration_simulation.png")))
print("SKIPPED French historical figure")


本 Notebook 不生成策略 P&L，也没有点时发行数据、逐笔价格、借券或成交记录。月末 t 的信号必须等数据可用后生成，并在下一个可执行时点交易；收益必须按两条腿一致的成交后区间计算。French 当前缓存不支持上述历史可执行性结论。